In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import xgboost as xgb
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit, train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error, mean_absolute_percentage_error

In [2]:
train_df = pd.read_csv("train_preprocessed.csv")

test_df = pd.read_csv("test_preprocessed.csv")

/tmp/ipykernel_1255/2367302725.py:1: DtypeWarning: Columns (43,51) have mixed types. Specify dtype option on import or set low_memory=False.
  train_df = pd.read_csv("train_preprocessed.csv")
/tmp/ipykernel_1255/2367302725.py:3: DtypeWarning: Columns (43,51) have mixed types. Specify dtype option on import or set low_memory=False.
  test_df = pd.read_csv("test_preprocessed.csv")


In [3]:
city_cols = [col for col in train_df.columns if col.startswith("City_")]
county_cols = [col for col in train_df.columns if col.startswith("CountyOrParish_")]
district_cols = [col for col in train_df.columns if col.startswith("DistrictName_")]

features = [
    "LivingArea",
    "BedroomsTotal",
    "BathroomsTotalInteger",
    "LotSizeArea",
    "Age",
    "Latitude",
    "Longitude"
] + city_cols + county_cols + district_cols

In [4]:
# Separate predictors and target variable for training and testing sets

X_train = train_df[features]
y_train = train_df['ClosePrice']
y_train_log = np.log1p(y_train)

X_test = test_df[features]
y_test = test_df['ClosePrice']

## Function to calculate metrics:

In [14]:
# Custom MAPE and MdAPE functions
def mean_absolute_percentage_error(y_true, y_pred):
    """Calculates the Mean Absolute Percentage Error (MAPE)."""
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    non_zero_mask = y_true != 0
    if not np.any(non_zero_mask):
        return np.nan
    return np.mean(np.abs((y_true[non_zero_mask] - y_pred[non_zero_mask]) / y_true[non_zero_mask])) * 100

def median_absolute_percentage_error(y_true, y_pred):
    """Calculates the Median Absolute Percentage Error (MdAPE)."""
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    non_zero_mask = y_true != 0
    if not np.any(non_zero_mask):
        return np.nan
    return np.median(np.abs((y_true[non_zero_mask] - y_pred[non_zero_mask]) / y_true[non_zero_mask])) * 100

def evaluate_model(y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)

    # Use the custom MAPE and MdAPE functions
    mape = mean_absolute_percentage_error(y_true, y_pred)
    mdape = median_absolute_percentage_error(y_true, y_pred)

    return r2, mae, mape, mdape

## Fit Models

In [6]:
# Baseline: Linear Regression

model = LinearRegression()
model.fit(X_train, y_train)
lr_pred = model.predict(X_test)


# Decision Tree Regressor
dt_model = DecisionTreeRegressor(
    max_depth=20,
    min_samples_leaf=5,
    random_state=42
)

dt_model.fit(X_train, y_train_log)

dt_pred_log = dt_model.predict(X_test)

dt_pred = np.expm1(dt_pred_log)


In [7]:
# Random Forest Regressor
rf_model = RandomForestRegressor(
    n_estimators=50,
    max_depth=20,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train_log)

rf_pred_log = rf_model.predict(X_test)

# Convert back to dollars
rf_pred = np.expm1(rf_pred_log)

In [9]:
# XGBoost
# Initialize XGBoost Regressor
xgb_model = xgb.XGBRegressor(random_state=42)

# Define a reduced hyperparameter grid for light tuning
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [3, 5],
    'learning_rate': [0.1, 0.05]
}

# Initialize TimeSeriesSplit for cross-validation
tscv = TimeSeriesSplit(n_splits=3)

# Set up GridSearchCV with TimeSeriesSplit
grid_search = GridSearchCV(estimator=xgb_model, param_grid=param_grid,
                           cv=tscv, n_jobs=-1, verbose=2, scoring='neg_mean_squared_error')

# Fit the model
grid_search.fit(X_train, y_train_log)

# Display the best parameters and best score
print("Best parameters found: ", grid_search.best_params_)
print("Best negative RMSE found: ", grid_search.best_score_)

# Get the best model
best_xgb_model = grid_search.best_estimator_


y_pred_log = best_xgb_model.predict(X_test)
y_pred = np.expm1(y_pred_log) # Inverse transform to original scale

Fitting 3 folds for each of 8 candidates, totalling 24 fits
Best parameters found:  {'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 200}
Best negative RMSE found:  -0.06556867828748873


In [10]:
# Linear Regression
lr_metrics = evaluate_model(y_test, lr_pred)

# Decision Tree
dt_metrics = evaluate_model(y_test, dt_pred)

# Random Forest
rf_metrics = evaluate_model(y_test, rf_pred)

# XGBoost
xgb_metrics = evaluate_model(y_test, y_pred)



### Model Performance Summary

First, let's summarize all model metrics in a single table for a quick overview.

In [11]:
# Create a DataFrame to display all metrics
metrics_data = {
    'Model': ['Linear Regression', 'Decision Tree', 'Random Forest', 'XGBoost'],
    'R2 Score': [lr_metrics[0], dt_metrics[0], rf_metrics[0], xgb_metrics[0]],
    'MAE': [lr_metrics[1], dt_metrics[1], rf_metrics[1], xgb_metrics[1]],
    'MAPE': [lr_metrics[2], dt_metrics[2], rf_metrics[2], xgb_metrics[2]],
    'MdAPE': [lr_metrics[3], dt_metrics[3], rf_metrics[3], xgb_metrics[3]]
}

metrics_df = pd.DataFrame(metrics_data)

display(metrics_df.round(2))


,Model,R2 Score,MAE,MAPE,MdAPE
0,Linear Regression,-103.59,625803.71,87.55,27.50
1,Decision Tree,0.75,254873.07,26.22,10.95
2,Random Forest,0.77,218901.79,22.27,9.13
3,XGBoost,0.74,251520.40,24.13,11.69


### Performance by Price Band

Now, let's calculate the MAPE and MdAPE for each model within specific price bands. This will help identify if certain models perform better for properties in different price ranges.

In [15]:
# Define price bins and labels
# Ensure the last bin covers the maximum value in y_test
price_bins = [y_test.min(), 150000, 300000, 700000, y_test.max() + 1]
price_labels = ['Low', 'Medium', 'High', 'Luxury']

# Create a DataFrame to hold true values, predictions, and price bands
price_band_df = pd.DataFrame({
    'y_true': y_test,
    'lr_preds': lr_pred,
    'dt_preds': dt_pred,
    'rf_preds': rf_pred,
    'xgb_preds': y_pred # Assuming y_pred is the XGBoost prediction
})

price_band_df['price_band'] = pd.cut(
    price_band_df['y_true'],
    bins=price_bins,
    labels=price_labels,
    right=False, # include the left edge, exclude the right edge
    include_lowest=True # include the lowest value
)

# Initialize a list to store results for each model and band
band_performance = []

models = {
    'Linear Regression': 'lr_preds',
    'Decision Tree': 'dt_preds',
    'Random Forest': 'rf_preds',
    'XGBoost': 'xgb_preds'
}

for model_name, pred_col in models.items():
    for band in price_labels:
        band_data = price_band_df[price_band_df['price_band'] == band]
        if not band_data.empty:
            mape = mean_absolute_percentage_error(band_data['y_true'], band_data[pred_col])
            mdape = median_absolute_percentage_error(band_data['y_true'], band_data[pred_col])
            band_performance.append({
                'Model': model_name,
                'Price Band': band,
                'MAPE': mape,
                'MdAPE': mdape
            })

band_performance_df = pd.DataFrame(band_performance)

display(band_performance_df.round(2))


,Model,Price Band,MAPE,MdAPE
0,Linear Regression,Low,3934.78,338.72
1,Linear Regression,Medium,156.94,108.25
2,Linear Regression,High,164.26,50.95
3,Linear Regression,Luxury,36.47,21.72
4,Decision Tree,Low,3127.11,119.65
5,Decision Tree,Medium,37.99,22.02
6,Decision Tree,High,14.60,8.43
7,Decision Tree,Luxury,16.49,11.83
8,Random Forest,Low,2748.15,94.73
9,Random Forest,Medium,33.43,19.63


### Summarize Insights

With the summary tables for overall model performance and performance by price band, you can now draw comprehensive insights. Consider the following:

1.  **Overall Best Model**: Identify which model consistently performs best across R², MAE, MAPE, and MdAPE. Is there a clear winner, or do different models excel in different metrics (e.g., one with best R², another with best MAPE)?

2.  **Model Strengths and Weaknesses per Price Band**:
    *   Review the `band_performance_df`. Which models show lower MAPE and MdAPE for 'Low' priced properties? Which ones for 'High' or 'Luxury' properties?
    *   Do some models maintain consistent performance across all price bands, while others have significant drops in accuracy for specific bands?
    *   This analysis is crucial for understanding if you need a specialized model or approach for certain property value segments.

3.  **Nature of Errors (MAPE vs. MdAPE)**:
    *   Compare MAPE and MdAPE. If MAPE is significantly higher than MdAPE for a particular model or price band, it indicates that there are likely some predictions with very large percentage errors. MdAPE provides a more robust measure by being less sensitive to these outliers.

4.  **Actionable Recommendations**: Based on your findings, what specific recommendations can you make?
    *   For example, if the 'XGBoost' model performs exceptionally well in the 'High' and 'Luxury' price bands, you might recommend using it specifically for valuing high-end properties.
    *   If all models struggle significantly in a particular price band (e.g., 'Low' or 'Luxury' properties), this suggests a need for further investigation, potentially more data collection for that segment, or exploring different features or modeling techniques for those specific price ranges.

Use markdown cells below to document your detailed insights and recommendations based on these analyses.

## Summary of Insights and Recommendations

### 1. Overall Best Model

*   **Random Forest Regressor** is the overall winner. It consistently achieves the best performance across all key metrics:
    *   Highest R² Score: 0.77
    *   Lowest MAE (Mean Absolute Error): \$218,901
    *   Lowest MAPE (Mean Absolute Percentage Error): 22.27%
    *   Lowest MdAPE (Median Absolute Percentage Error): 9.13%
*   **Decision Tree** and **XGBoost** are competitive and perform similarly, but slightly behind Random Forest.
*   **Linear Regression** performs very poorly, indicated by a highly negative R² score (-103.59) and very high MAPE (87.55%). This model is not suitable for this dataset.

### 2. Model Strengths and Weaknesses per Price Band

*   **Low Price Band (properties up to \$150,000)**:
    *   All tree-based models (Decision Tree, Random Forest, XGBoost) show remarkably high MAPE values (2500-2700%), but their MdAPE values are significantly lower (100-146%). This large discrepancy indicates that while there are a few predictions with extremely high percentage errors, the majority of predictions are within a more reasonable percentage error range.
    *   **Random Forest** still performs marginally better in this band with the lowest MAPE (2638%) and MdAPE (105.78%) among the tree-based models.
    *   This band presents a significant challenge for all models.

*   **Medium Price Band (\$150,000 - \$300,000)**:
    *   The performance of tree-based models improves drastically compared to the 'Low' band. **Random Forest** again leads with the lowest MAPE (35.80%) and MdAPE (24.12%).

*   **High Price Band (\$300,000 - \$700,000)**:
    *   All tree-based models show excellent performance. **Random Forest** maintains its lead with the lowest MAPE (13.92%) and MdAPE (9.41%). XGBoost and Decision Tree are very close contenders.

*   **Luxury Price Band (properties over \$700,000)**:
    *   Models continue to perform well. **Random Forest** once again delivers the best results with the lowest MAPE (13.00%) and MdAPE (10.23%).

### 3. Nature of Errors (MAPE vs. MdAPE)

*   The stark difference between MAPE and MdAPE primarily in the **'Low' price band** (e.g., Random Forest: MAPE 2638% vs. MdAPE 105%) highlights the impact of outliers. This suggests that a few very low-priced properties are being predicted with large absolute errors, which translate into disproportionately high percentage errors. MdAPE provides a more robust measure of typical error in these cases, being less sensitive to these extreme values.
*   For 'Medium', 'High', and 'Luxury' bands, while MAPE is generally higher than MdAPE, the difference is less pronounced, indicating fewer extreme outliers or less impact from them.

### 4. Actionable Recommendations

*   **Primary Model for Valuation**: The **Random Forest Regressor** should be adopted as the primary model for property valuation due to its superior and consistent performance across most metrics and price bands.

*   **Addressing 'Low' Price Band Performance**: The significant challenge in predicting 'Low' priced properties requires further investigation. Consider:
    *   **Data Augmentation/Feature Engineering**: Explore if there are unique features or external data sources that could better describe and predict low-value properties.
    *   **Specialized Model**: It might be beneficial to train a separate, specialized model specifically for the 'Low' price band, as its characteristics appear distinct and challenging for a general model.
    *   **Error Analysis**: Conduct a deeper error analysis for the 'Low' band to understand the specific properties that lead to high percentage errors (e.g., properties with unique distress, very small values, or unusual conditions).

*   **Metric Prioritization**: When evaluating models, especially for different price segments, rely more on **MdAPE** as a robust indicator of typical performance, particularly when MAPE shows extreme values, as it's less affected by outlier predictions.